In [ ]:

# CELL 1 — Setup and validate uploaded dataset ZIP

from pathlib import Path
import zipfile
import pandas as pd
import json
import shutil

from google.colab import files

uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_files) != 1:
    raise ValueError(f"Upload exactly one dataset ZIP. Found: {zip_files}")

ZIP_PATH = Path(zip_files[0])
EXTRACT_DIR = Path("/content/dataset_splits_audit")

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The uploaded file is not a valid ZIP archive.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad_file = z.testzip()
    if bad_file:
        raise ValueError(f"Corrupt file inside ZIP: {bad_file}")

    # Extract safely into the target directory.
    for member in z.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if target != EXTRACT_DIR.resolve() and EXTRACT_DIR.resolve() not in target.parents:
            raise ValueError(f"Unsafe ZIP path: {member.filename}")
    z.extractall(EXTRACT_DIR)

# Locate the CSVs even if the ZIP contains a nested folder.
def find_file(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(matches)}"
        )
    return matches[0]

TRAIN_PATH = find_file("train.csv")
VAL_PATH = find_file("validation.csv")
TEST_PATH = find_file("test.csv")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

LABELS = [
    "as-IN", "bn-IN", "en-IN", "gu-IN", "hi-IN", "kn-IN",
    "ml-IN", "mr-IN", "pa-IN", "ta-IN", "te-IN", "none"
]

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    required = {"text", "final_label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name}.csv is missing columns: {missing}")

    if df["text"].isna().any() or df["final_label"].isna().any():
        raise ValueError(f"{name}.csv contains missing text or labels.")

    unknown = set(df["final_label"].astype(str).unique()) - set(LABELS)
    if unknown:
        raise ValueError(f"{name}.csv has unexpected labels: {unknown}")

    print(f"\n{name.upper()}: {len(df):,} rows")
    print(df["final_label"].value_counts().reindex(LABELS, fill_value=0))

print("\nAll three CSVs loaded and label checks passed.")
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)

print("\nNext: train one shared multilingual tokenizer on TRAIN text only,")
print("then adapt and fine-tune E2 and E4 using the same tokenizer.")


Saving dataset_clean_v2.zip to dataset_clean_v2.zip

TRAIN: 374,860 rows
final_label
as-IN    35789
bn-IN    35716
en-IN    22538
gu-IN    32187
hi-IN    38307
kn-IN    32200
ml-IN    32173
mr-IN    36750
pa-IN    28620
ta-IN    32264
te-IN    32170
none     16146
Name: count, dtype: int64

VALIDATION: 47,117 rows
final_label
as-IN    4574
bn-IN    4560
en-IN    2955
gu-IN    3970
hi-IN    4737
kn-IN    4031
ml-IN    4082
mr-IN    4454
pa-IN    3627
ta-IN    3981
te-IN    4080
none     2066
Name: count, dtype: int64

TEST: 46,855 rows
final_label
as-IN    4498
bn-IN    4570
en-IN    2923
gu-IN    4066
hi-IN    4730
kn-IN    3960
ml-IN    3936
mr-IN    4637
pa-IN    3574
ta-IN    3947
te-IN    3941
none     2073
Name: count, dtype: int64

All three CSVs loaded and label checks passed.
Train: /content/dataset_splits_audit/train.csv
Validation: /content/dataset_splits_audit/validation.csv
Test: /content/dataset_splits_audit/test.csv

Next: train one shared multilingual tokenizer on TRAIN 

Train multilingual WordPiece tokenizer

In [ ]:
df=pd.read_csv("/content/dataset_splits_audit/train.csv")

In [ ]:
df.groupby(['label','category']).count()

text  source  source_file  \
label category                                                                
as-IN as_stt_codemix                              3849    3849         3849   
      assamese_english_mixed_script              16048   16048        16048   
      assamese_only                              15892   15892        15892   
bn-IN bengali_english_mixed_script               15948   15948        15948   
      bengali_only                               15908   15908        15908   
      bn_stt_codemix                              3860    3860         3860   
en-IN en_stt_codemix                               827     827          827   
      english_only                               21711   21711        21711   
gu-IN gu_stt_codemix                               183     183          183   
      gujarati_english_mixed_script              16028   16028        16028   
      gujarati_only                              15976   15976        15976   
hi-IN hi_stt_codemix                              6224    6224         6224   
      hindi_english_mixed_script                 16033   16033        16033   
      hindi_only                                 16050   16050        16050   
kn-IN kannada_english_mixed_script               16028   16028        16028   
      kannada_only                               16015   16015        16015   
      kn_stt_codemix                               157     157          157   
ml-IN malayalam_english_mixed_script             16021   16021        16021   
      malayalam_only                             15998   15998        15998   
      ml_stt_codemix                               154     154          154   
mr-IN marathi_english_mixed_script               16019   16019        16019   
      marathi_only                               16026   16026        16026   
      mr_stt_codemix                              4705    4705         4705   
none  none_stt_codemix                             217     217          217   
      unsupported_language_english_mixed_script   7952    7952         7952   
      unsupported_language_only                   7977    7977         7977   
pa-IN pa_stt_codemix                              4646    4646         4646   
      punjabi_english_mixed_script                8027    8027         8027   
      punjabi_hindi_english_mixed_script             8       8            8   
      punjabi_only                               15939   15939        15939   
ta-IN ta_stt_codemix                               154     154          154   
      tamil_english_mixed_script                 16025   16025        16025   
      tamil_only                                 16085   16085        16085   
te-IN te_stt_codemix                               155     155          155   
      telugu_english_mixed_script                16039   16039        16039   
      telugu_only                                15976   15976        15976   

                                                 source_group_id  text_sha256  \
label category                                                                  
as-IN as_stt_codemix                                        3849         3849   
      assamese_english_mixed_script                        16048        16048   
      assamese_only                                        15892        15892   
bn-IN bengali_english_mixed_script                         15948        15948   
      bengali_only                                         15908        15908   
      bn_stt_codemix                                        3860         3860   
en-IN en_stt_codemix                                         827          827   
      english_only                                         21711        21711   
gu-IN gu_stt_codemix                                         183          183   
      gujarati_english_mixed_script                        16028        16028   
      gujarati_only                                        15976        159

In [ ]:
df[df['category']=='mr_stt_codemix']

,text,label,category,source,source_file,source_group_id,text_sha256,review_status,is_mixed_script_candidate,is_synthetic,generator,template_id,final_label,group_id,split
23,form कधीपर्यंत मिळेल लगेच,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p1:form,b09eb65542781628e94a14d6a861c2b9f275ca1e27a0b1...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p1:form,mr-IN,stt:mr:p1:form,train
38,"ठीक आहे, मी statement बद्दल याआधीही विचारलं हो...",mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p10:statement,8c82ec9b0d40b466fc2e93d1d27a60b74742893785db92...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p10:statement,mr-IN,stt:mr:p10:statement,train
50,"मग, मला plan मध्ये बदल करायचा आहे",mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p6:plan,582c2ab0af7306a6e2de66a20dbb72ce8b6e434ac37aa2...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p6:plan,mr-IN,stt:mr:p6:plan,train
157,password साठी दुसरा पर्याय आहे का ना,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p8:password,c1d7399ca9c5f59bb76b15e7131a6bcebb45dfd012aa86...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p8:password,mr-IN,stt:mr:p8:password,train
260,मी मेसेज बद्दल याआधीही विचारलं होतं परत,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p10:message,1af790d00990c85fa91e8f566981344e441fa6887598a4...,synthetic_unreviewed,False,True,stt_style_v1,stt:mr:p10:message,mr-IN,stt:mr:p10:message,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374489,अच्छा मला registration ची गरज आहे परत,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p9:registration,ccbc4e77abcc6d57ee89354c03be2c2f2f7ad10e4257f7...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p9:registration,mr-IN,stt:mr:p9:registration,train
374505,मॅडम तुम्ही माझी SIM तपासून पाहाल का लगेच,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p2:SIM,2229b9b0863f133aa066566278e3279526ec0a0f7817b0...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p2:SIM,mr-IN,stt:mr:p2:SIM,train
374624,एक मिनिट request साठी दुसरा पर्याय आहे का,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p8:request,db441da6ec292f3c671822b2fc0adce90e9ca7cf50a362...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p8:request,mr-IN,stt:mr:p8:request,train
374661,तर मला complaint ची गरज आहे,mr-IN,mr_stt_codemix,generated/stt_style_v1,gen_stt.py,stt:mr:p9:complaint,3e4e1a53254cec5d15dbc3dfcd82f5a6f58dd41d50a11b...,synthetic_unreviewed,True,True,stt_style_v1,stt:mr:p9:complaint,mr-IN,stt:mr:p9:complaint,train


In [ ]:

# CELL 2 — Train shared multilingual WordPiece tokenizer

!pip -q install -U tokenizers transformers

from pathlib import Path
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizerFast
import json
import numpy as np
import unicodedata

TOKENIZER_DIR = Path("/content/multilingual_wordpiece_tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 30522
MIN_FREQUENCY = 2
MAX_LENGTH = 128

# Use training data ONLY.
# Do not include validation or test text.
training_texts = train_df["text"].dropna().astype(str)

def text_iterator():
    for text in training_texts:
        yield unicodedata.normalize("NFKC", text)

print(f"Training tokenizer on {len(training_texts):,} sentences")

tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=False
)

tokenizer.train_from_iterator(
    text_iterator(),
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[MASK]"
    ],
    limit_alphabet=6000,
    wordpieces_prefix="##"
)

tokenizer.save_model(str(TOKENIZER_DIR))

hf_tokenizer = BertTokenizerFast(
    vocab_file=str(TOKENIZER_DIR / "vocab.txt"),
    do_lower_case=False,
    strip_accents=False,
    tokenize_chinese_chars=True,
    model_max_length=MAX_LENGTH
)

hf_tokenizer.save_pretrained(str(TOKENIZER_DIR))

print("\nTokenizer training complete.")
print("Actual vocabulary size:", len(hf_tokenizer))
print("Tokenizer directory:", TOKENIZER_DIR)

samples = [
    "Eligibility क्या है?",
    "मुझे admission process बताओ",
    "What is the course fee?",
    "ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों",
    "मराठी आणि English mix",
    "తెలుగు మరియు English",
    "தமிழ் and English",
    "বাংলা and English",
    "ગુજરાતી and English",
    "ಕನ್ನಡ and English",
    "മലയാളം and English",
    "অসমীয়া and English"
]

print("\nTokenization examples:\n")

for text in samples:
    tokens = hf_tokenizer.tokenize(text)
    ids = hf_tokenizer.encode(text, add_special_tokens=True)
    print("TEXT:", text)
    print("TOKENS:", tokens)
    print("TOKEN COUNT:", len(ids))
    print("-" * 55)

# Audit unknown-token frequency and sequence lengths
rng = np.random.default_rng(42)
sample_size = min(10000, len(training_texts))
indices = rng.choice(len(training_texts), sample_size, replace=False)

unk_count = 0
total_tokens = 0
lengths = []

for idx in indices:
    ids = hf_tokenizer.encode(
        training_texts.iloc[int(idx)],
        add_special_tokens=False,
        truncation=False
    )
    unk_count += ids.count(hf_tokenizer.unk_token_id)
    total_tokens += len(ids)
    lengths.append(len(ids) + 2)

print("\nTOKENIZER AUDIT")
print("Sample sentences:", sample_size)
print("UNK token rate:", round(100 * unk_count / max(total_tokens, 1), 4), "%")
print("Average sequence length:", round(float(np.mean(lengths)), 2))
print("P50 length:", int(np.percentile(lengths, 50)))
print("P90 length:", int(np.percentile(lengths, 90)))
print("P95 length:", int(np.percentile(lengths, 95)))
print("P99 length:", int(np.percentile(lengths, 99)))
print(
    "Percentage exceeding max length:",
    round(100 * np.mean(np.array(lengths) > MAX_LENGTH), 2),
    "%"
)

with open(TOKENIZER_DIR / "tokenizer_audit.json", "w") as f:
    json.dump({
        "vocab_size": len(hf_tokenizer),
        "training_rows": len(training_texts),
        "sample_size": sample_size,
        "unk_rate_percent": 100 * unk_count / max(total_tokens, 1),
        "mean_sequence_length": float(np.mean(lengths)),
        "p50": float(np.percentile(lengths, 50)),
        "p90": float(np.percentile(lengths, 90)),
        "p95": float(np.percentile(lengths, 95)),
        "p99": float(np.percentile(lengths, 99)),
        "max_length": MAX_LENGTH,
        "fraction_over_max_length": float(
            np.mean(np.array(lengths) > MAX_LENGTH)
        )
    }, f, indent=2)

print("\nTokenizer saved successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.6 MB/s eta 0:00:00
Training tokenizer on 374,860 sentences

Tokenizer training complete.
Actual vocabulary size: 5
Tokenizer directory: /content/multilingual_wordpiece_tokenizer

Tokenization examples:

TEXT: Eligibility क्या है?
TOKENS: ['[UNK]', '[UNK]', '[UNK]', '[UNK]']
TOKEN COUNT: 6
-------------------------------------------------------
TEXT: मुझे admission process बताओ
TOKENS: ['[UNK]', '[UNK]', '[UNK]', '[UNK]']
TOKEN COUNT: 6
-------------------------------------------------------
TEXT: What is the course fee?
TOKENS: ['[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]']
TOKEN COUNT: 8
-------------------------------------------------------
TEXT: ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों
TOKENS: ['[UNK]', '[UNK]', '[UNK]', '[UNK]']
TOKEN COUNT: 6
-------------------------------------------------------
TEXT: मराठी आणि English mix
TOKENS: ['[UNK]', '[UNK]', '[

In [ ]:

# FIXED CELL 2 — Train a multilingual WordPiece tokenizer

!pip -q install -U tokenizers transformers

from pathlib import Path
import unicodedata
import json
import numpy as np

from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

TOKENIZER_DIR = Path("/content/multilingual_wordpiece_tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 30522
MIN_FREQUENCY = 2
MAX_LENGTH = 128

assert "train_df" in globals(), "Run the dataset loading cell first."
assert "text" in train_df.columns, "train_df must contain a text column."

training_texts = (
    train_df["text"].dropna().astype(str).tolist()
)

assert len(training_texts) > 0, "No training text found."

def text_iterator():
    for text in training_texts:
        text = unicodedata.normalize("NFKC", text)
        if text.strip():
            yield text

special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

# Create a fresh native WordPiece tokenizer.
backend = Tokenizer(models.WordPiece(unk_token="[UNK]"))

backend.normalizer = normalizers.BertNormalizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=False
)
backend.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
backend.decoder = decoders.WordPiece(prefix="##")

trainer = trainers.WordPieceTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=special_tokens,
    limit_alphabet=6000,
    continuing_subword_prefix="##",
    show_progress=True
)

print(f"Training on {len(training_texts):,} training sentences...")

backend.train_from_iterator(
    text_iterator(),
    trainer=trainer,
    length=len(training_texts)
)

actual_vocab_size = backend.get_vocab_size()

print("Actual vocabulary size:", actual_vocab_size)

# Fail safely rather than accepting a five-token vocabulary.
if actual_vocab_size < 1000:
    raise RuntimeError(
        f"Tokenizer vocabulary is only {actual_vocab_size}. "
        "Training failed; do not fine-tune the models."
    )

# Save native tokenizer and vocab.
backend.save(str(TOKENIZER_DIR / "tokenizer.json"))
backend.model.save(str(TOKENIZER_DIR))

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    model_max_length=MAX_LENGTH
)
hf_tokenizer.save_pretrained(str(TOKENIZER_DIR))

# Validate using real multilingual examples.
samples = [
    "Eligibility क्या है?",
    "मुझे admission process बताओ",
    "What is the course fee?",
    "ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों",
    "मराठी आणि English mix",
    "తెలుగు మరియు English",
    "தமிழ் and English",
    "বাংলা and English",
    "ગુજરાતી and English",
    "ಕನ್ನಡ and English",
    "മലയാളം and English",
    "অসমীয়া and English"
]

print("\nMULTILINGUAL TOKENIZATION CHECK")
for text in samples:
    tokens = hf_tokenizer.tokenize(text)
    print(f"\n{text}")
    print(tokens)

# Audit unknown-token rate on a reproducible training sample.
rng = np.random.default_rng(42)
sample_size = min(10000, len(training_texts))
indices = rng.choice(len(training_texts), sample_size, replace=False)

unk_id = hf_tokenizer.unk_token_id
unk_count = 0
total_tokens = 0
lengths = []

for idx in indices:
    ids = hf_tokenizer(
        training_texts[int(idx)],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    unk_count += sum(token_id == unk_id for token_id in ids)
    total_tokens += len(ids)
    lengths.append(len(ids) + 2)  # [CLS] and [SEP]

unk_rate = 100 * unk_count / max(total_tokens, 1)

print("\nTOKENIZER AUDIT")
print("Vocabulary size:", len(hf_tokenizer))
print("Sample sentences:", sample_size)
print(f"UNK token rate: {unk_rate:.4f}%")
print(f"Average sequence length: {np.mean(lengths):.2f}")
print(f"P50: {np.percentile(lengths, 50):.0f}")
print(f"P90: {np.percentile(lengths, 90):.0f}")
print(f"P95: {np.percentile(lengths, 95):.0f}")
print(f"P99: {np.percentile(lengths, 99):.0f}")
print(
    f"Over {MAX_LENGTH} tokens: "
    f"{100 * np.mean(np.array(lengths) > MAX_LENGTH):.2f}%"
)

audit = {
    "vocab_size": len(hf_tokenizer),
    "training_rows": len(training_texts),
    "sample_size": sample_size,
    "unk_rate_percent": unk_rate,
    "mean_sequence_length": float(np.mean(lengths)),
    "p50": float(np.percentile(lengths, 50)),
    "p90": float(np.percentile(lengths, 90)),
    "p95": float(np.percentile(lengths, 95)),
    "p99": float(np.percentile(lengths, 99)),
    "max_length": MAX_LENGTH
}

with open(TOKENIZER_DIR / "tokenizer_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("\nTokenizer saved to:", TOKENIZER_DIR)
print("Validation passed.")


Training on 374,860 training sentences...
Actual vocabulary size: 30522

MULTILINGUAL TOKENIZATION CHECK

Eligibility क्या है?
['El', '##ig', '##ibility', 'क्या', 'है', '?']

मुझे admission process बताओ
['मुझे', 'admission', 'process', 'बता', '##ओ']

What is the course fee?
['Wh', '##at', 'is', 'the', 'course', 'fee', '?']

ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों
['ਪੰ', '##ਜਾ', '##ਬੀ', 'ਅਤੇ', 'हिंदी', 'दोनों']

मराठी आणि English mix
['म', '##रा', '##ठी', 'आणि', 'English', 'mix']

తెలుగు మరియు English
['తెలు', '##గు', 'మరియు', 'English']

தமிழ் and English
['தமிழ்', 'and', 'English']

বাংলা and English
['বা', '##ং', '##লা', 'and', 'English']

ગુજરાતી and English
['ગુજરાત', '##ી', 'and', 'English']

ಕನ್ನಡ and English
['ಕನ', '##್', '##ನ', '##ಡ', 'and', 'English']

മലയാളം and English
['മല', '##യ', '##ാളം', 'and', 'English']

অসমীয়া and English
['অসম', '##ীয়া', 'and', 'English']


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (134 > 128). Running this sequence through the model will result in indexing errors



TOKENIZER AUDIT
Vocabulary size: 30522
Sample sentences: 10000
UNK token rate: 0.1712%
Average sequence length: 30.79
P50: 29
P90: 50
P95: 58
P99: 74
Over 128 tokens: 0.01%

Tokenizer saved to: /content/multilingual_wordpiece_tokenizer
Validation passed.


Fine tuning bert tiny E2

In [ ]:

# CELL 3 — Fine-tune BERT-Tiny E2
# Requires: train_df, val_df, test_df from Cell 1
#           multilingual tokenizer from Cell 2

!pip -q install -U transformers scikit-learn tqdm

import os
import json
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer as OriginalTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------

SEED = 42
CHECKPOINT = "google/bert_uncased_L-2_H-128_A-2"
TOKENIZER_DIR = "/content/multilingual_wordpiece_tokenizer"
OUTPUT_DIR = Path("/content/bert_tiny_e2_multilingual_version4")

LABELS = [
    "as-IN", "bn-IN", "en-IN", "gu-IN", "hi-IN", "kn-IN",
    "ml-IN", "mr-IN", "pa-IN", "ta-IN", "te-IN", "none"
]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MAX_LENGTH = 128
BATCH_SIZE = 64
EPOCHS = 3
ENCODER_LR = 5e-5
EMBEDDING_LR = 3e-4
CLASSIFIER_LR = 3e-4
WEIGHT_DECAY = 0.01

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab, select Runtime > Change runtime type > GPU."
    )

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

for name, frame in [
    ("train", train_df), ("validation", val_df), ("test", test_df)
]:
    if not {"text", "final_label"}.issubset(frame.columns):
        raise ValueError(f"{name} dataframe is missing required columns.")
    if not set(frame["final_label"].unique()).issubset(set(LABELS)):
        raise ValueError(f"Unexpected labels in {name} dataframe.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------- LOAD TOKENIZERS ----------------

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_DIR,
    use_fast=True
)

# Original tokenizer is used ONLY to transfer matching pretrained embeddings.
original_tokenizer = OriginalTokenizer.from_pretrained(CHECKPOINT)

assert len(tokenizer) == 30522, (
    f"Expected vocabulary size 30522, got {len(tokenizer)}"
)

# ---------------- LOAD PRETRAINED E2 ----------------

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

assert model.config.num_hidden_layers == 2
assert model.config.hidden_size == 128
assert model.config.num_attention_heads == 2

# ---------------- ADAPT TOKEN EMBEDDINGS ----------------
# New token IDs do not align with original English vocabulary IDs.
# Initialize all rows, then transfer embeddings for matching tokens.
# For Latin tokens, try lowercase matching as well.
# Unmatched Indic tokens receive random initialization.

old_vocab = original_tokenizer.get_vocab()
new_vocab = tokenizer.get_vocab()

old_embedding = (
    model.bert.embeddings.word_embeddings.weight.detach().clone()
)
new_embedding = torch.empty_like(old_embedding)

nn.init.normal_(
    new_embedding,
    mean=0.0,
    std=model.config.initializer_range
)

matched = 0
exact_matches = 0
lowercase_matches = 0

for token, new_id in new_vocab.items():
    old_id = old_vocab.get(token)

    if old_id is not None:
        new_embedding[new_id] = old_embedding[old_id]
        matched += 1
        exact_matches += 1
        continue

    lower_token = token.lower()
    old_id = old_vocab.get(lower_token)

    if old_id is not None:
        new_embedding[new_id] = old_embedding[old_id]
        matched += 1
        lowercase_matches += 1

# Keep padding embedding zero.
if tokenizer.pad_token_id is not None:
    new_embedding[tokenizer.pad_token_id].zero_()

with torch.no_grad():
    model.bert.embeddings.word_embeddings.weight.copy_(new_embedding)

print("\nEmbedding adaptation:")
print("New vocabulary:", len(new_vocab))
print("Exact pretrained token matches:", exact_matches)
print("Lowercase pretrained token matches:", lowercase_matches)
print("Total transferred rows:", matched)
print("Randomly initialized rows:", len(new_vocab) - matched)
print("Transfer coverage:", round(100 * matched / len(new_vocab), 2), "%")

# ---------------- DATASET ----------------

class TextDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["text"].astype(str).tolist()
        self.labels = [
            LABEL2ID[x] for x in frame["final_label"].astype(str).tolist()
        ]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded

train_loader = DataLoader(
    TextDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    TextDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    TextDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

# ---------------- OPTIMIZER ----------------

embedding_params = []
classifier_params = []
encoder_params = []

for name, param in model.named_parameters():
    if "embeddings.word_embeddings" in name:
        embedding_params.append(param)
    elif name.startswith("classifier."):
        classifier_params.append(param)
    else:
        encoder_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_params,
            "lr": ENCODER_LR,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": embedding_params,
            "lr": EMBEDDING_LR,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": classifier_params,
            "lr": CLASSIFIER_LR,
            "weight_decay": WEIGHT_DECAY
        }
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.05 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=True)
model.to(device)

# ---------------- EVALUATION FUNCTION ----------------

def evaluate(loader):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**batch)

            losses.append(outputs.loss.item())
            preds = outputs.logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_targets.extend(batch["labels"].cpu().numpy().tolist())

    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy_score(all_targets, all_preds)),
        "macro_f1": float(
            f1_score(
                all_targets,
                all_preds,
                labels=list(range(len(LABELS))),
                average="macro",
                zero_division=0
            )
        ),
        "predictions": all_preds,
        "targets": all_targets
    }

# ---------------- TRAIN ----------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_macro_f1 = -1.0
history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []
    start_time = time.time()

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:
        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_losses.append(loss.item())
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    val_metrics = evaluate(val_loader)

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(epoch_losses)),
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"],
        "validation_macro_f1": val_metrics["macro_f1"],
        "seconds": round(time.time() - start_time, 1)
    }
    history.append(epoch_result)

    print("\nEpoch result:")
    print(json.dumps(epoch_result, indent=2))

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        with open(OUTPUT_DIR / "label_mapping.json", "w") as f:
            json.dump(
                {"label2id": LABEL2ID, "id2label": ID2LABEL},
                f,
                indent=2
            )

        print("Saved new best checkpoint:", OUTPUT_DIR)

# Save training history.
with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

# ---------------- FINAL TEST EVALUATION ----------------
# Test data is evaluated only after selecting the best checkpoint by validation macro-F1.

del model
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.to(device)

test_metrics = evaluate(test_loader)

print("\nFINAL TEST RESULTS")
print("Accuracy:", round(test_metrics["accuracy"], 4))
print("Macro F1:", round(test_metrics["macro_f1"], 4))
print("Test loss:", round(test_metrics["loss"], 4))

report = classification_report(
    test_metrics["targets"],
    test_metrics["predictions"],
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

report_df.to_csv(OUTPUT_DIR / "test_classification_report.csv")

cm = confusion_matrix(
    test_metrics["targets"],
    test_metrics["predictions"],
    labels=list(range(len(LABELS)))
)

pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
).to_csv(OUTPUT_DIR / "test_confusion_matrix.csv")

with open(OUTPUT_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "checkpoint": CHECKPOINT,
        "architecture": "BERT-Tiny E2",
        "labels": LABELS,
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "test_loss": test_metrics["loss"],
        "embedding_transfer_coverage": matched / len(new_vocab),
        "history": history
    }, f, indent=2)

# Zip final artifacts for download.
import shutil
zip_path = shutil.make_archive(
    "/content/bert_tiny_e2_multilingual",
    "zip",
    OUTPUT_DIR
)

print("\nModel artifacts:", zip_path)
print("Best validation macro F1:", round(best_macro_f1, 4))

from google.colab import files
files.download(zip_path)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.8 MB/s eta 0:00:00
GPU: Tesla T4
PyTorch: 2.11.0+cu130


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

...


FINAL TEST RESULTS
Accuracy: 0.9997
Macro F1: 0.9997
Test loss: 0.0016


,precision,recall,f1-score,support
as-IN,0.999555,0.998666,0.999110,4498.00000
bn-IN,0.998688,0.999562,0.999125,4570.00000
en-IN,0.998633,0.999658,0.999145,2923.00000
gu-IN,1.000000,1.000000,1.000000,4066.00000
hi-IN,1.000000,0.999789,0.999894,4730.00000
kn-IN,1.000000,0.999747,0.999874,3960.00000
ml-IN,1.000000,1.000000,1.000000,3936.00000
mr-IN,0.999353,0.999784,0.999569,4637.00000
pa-IN,1.000000,1.000000,1.000000,3574.00000
ta-IN,1.000000,0.999747,0.999873,3947.00000



Model artifacts: /content/bert_tiny_e2_multilingual.zip
Best validation macro F1: 0.9997


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FINE TINING BERT E4

In [ ]:
from pathlib import Path

for p in Path("/content").rglob("*.csv"):
    print(p)

for p in Path("/content").rglob("*.zip"):
    print("ZIP:", p)

/content/dataset_splits_audit/leakage_audit.csv
/content/dataset_splits_audit/label_distribution.csv
/content/dataset_splits_audit/validation.csv
/content/dataset_splits_audit/test.csv
/content/dataset_splits_audit/category_label_mapping.csv
/content/dataset_splits_audit/train.csv
/content/bert_tiny_e2_multilingual/test_confusion_matrix.csv
/content/bert_tiny_e2_multilingual/test_classification_report.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
ZIP: /content/dataset_splits_audit1 (1).zip
ZIP: /content/bert_tiny_e2_multilingual.zip
ZIP: /content/dataset_splits_audit1.zip


In [ ]:
# Install dependencies if needed
!pip -q install transformers tokenizers scikit-learn pandas numpy

import os, json, time, random, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ---------------- CONFIG ----------------
SEED = 42
CHECKPOINT = "google/bert_uncased_L-4_H-128_A-2"

DATA_DIR = Path("/content/dataset_splits_audit")
TOKENIZER_DIR = Path("/content/multilingual_wordpiece_tokenizer")
OUTPUT_DIR = Path("/content/bert_tiny_e4_multilingual")

MAX_LENGTH = 128
BATCH_SIZE = 64
EPOCHS = 3
ENCODER_LR = 5e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 0.01

assert torch.cuda.is_available(), "Enable GPU in Colab: Runtime > Change runtime type > T4 GPU"
assert TOKENIZER_DIR.exists(), f"Tokenizer missing: {TOKENIZER_DIR}"

for filename in ["train.csv", "validation.csv", "test.csv"]:
    assert (DATA_DIR / filename).exists(), f"Missing: {DATA_DIR / filename}"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

# ---------------- LOAD DATA ----------------
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "validation.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

required_cols = {"text", "label"}
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing = required_cols - set(df.columns)
    assert not missing, f"{name}.csv missing columns: {missing}"

train_df = train_df.dropna(subset=["text", "label"]).copy()
val_df   = val_df.dropna(subset=["text", "label"]).copy()
test_df  = test_df.dropna(subset=["text", "label"]).copy()

train_df["text"] = train_df["text"].astype(str)
val_df["text"] = val_df["text"].astype(str)
test_df["text"] = test_df["text"].astype(str)

labels = sorted(train_df["label"].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

for name, df in [("validation", val_df), ("test", test_df)]:
    unseen = set(df["label"].unique()) - set(labels)
    assert not unseen, f"Labels in {name} not present in train: {unseen}"

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Labels:", label2id)

# ---------------- TOKENIZER ----------------
tokenizer = AutoTokenizer.from_pretrained(
    str(TOKENIZER_DIR),
    use_fast=True
)
print("Tokenizer vocabulary:", len(tokenizer))

# ---------------- LOAD E4 ----------------
config = AutoConfig.from_pretrained(CHECKPOINT)
assert config.num_hidden_layers == 4, config.num_hidden_layers
assert config.hidden_size == 128, config.hidden_size

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# Transfer compatible pretrained embedding rows into custom vocabulary.
# All other custom-token rows retain random initialization.
old_vocab = model.get_input_embeddings().weight.detach().clone()
old_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True)

new_vocab_size = len(tokenizer)
embedding_dim = old_vocab.shape[1]

new_embeddings = torch.empty(
    new_vocab_size, embedding_dim, dtype=old_vocab.dtype
)
torch.nn.init.normal_(
    new_embeddings,
    mean=0.0,
    std=config.initializer_range
)

old_vocab_dict = old_tokenizer.get_vocab()
new_vocab_dict = tokenizer.get_vocab()

transferred_exact = 0
transferred_lower = 0

for token, new_id in new_vocab_dict.items():
    old_id = old_vocab_dict.get(token)

    if old_id is not None:
        new_embeddings[new_id] = old_vocab[old_id]
        transferred_exact += 1
    elif token.lower() in old_vocab_dict:
        new_embeddings[new_id] = old_vocab[old_vocab_dict[token.lower()]]
        transferred_lower += 1

model.resize_token_embeddings(new_vocab_size)
with torch.no_grad():
    model.get_input_embeddings().weight.copy_(new_embeddings)

transferred = transferred_exact + transferred_lower
print(f"Exact embedding transfers: {transferred_exact}")
print(f"Lowercase fallback transfers: {transferred_lower}")
print(f"Total transferred: {transferred}/{new_vocab_size} "
      f"({100*transferred/new_vocab_size:.2f}%)")
print(f"Randomly initialized token rows: {new_vocab_size-transferred}")

# ---------------- DATASET ----------------
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.targets = [label2id[x] for x in df["label"].tolist()]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item["labels"] = torch.tensor(self.targets[idx], dtype=torch.long)
        return item

train_loader = DataLoader(
    TextDataset(train_df), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    TextDataset(val_df), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    TextDataset(test_df), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

# ---------------- OPTIMIZER ----------------
model.to(device)

embedding_params = list(model.get_input_embeddings().parameters())
embedding_param_ids = {id(p) for p in embedding_params}
encoder_params = [
    p for n, p in model.named_parameters()
    if not n.startswith("classifier.")
    and id(p) not in embedding_param_ids
]
head_params = list(model.classifier.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": embedding_params + head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler("cuda")

# ---------------- EVALUATION ----------------
@torch.no_grad()
def evaluate(loader):
    model.eval()
    losses, predictions, targets = [], [], []

    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**batch)

        losses.append(outputs.loss.item())
        predictions.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
        targets.extend(batch["labels"].cpu().tolist())

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro"),
        "predictions": predictions,
        "targets": targets,
    }

# ---------------- TRAIN ----------------
best_val_f1 = -1
history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    start = time.time()

    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item()

        if step % 500 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | "
                  f"Step {step}/{len(train_loader)} | "
                  f"Loss {loss.item():.4f}")

    val_metrics = evaluate(val_loader)
    elapsed = time.time() - start

    row = {
        "epoch": epoch + 1,
        "train_loss": epoch_loss / len(train_loader),
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "seconds": elapsed,
    }
    history.append(row)
    print(json.dumps(row, indent=2))

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
        tokenizer.save_pretrained(OUTPUT_DIR)
        with open(OUTPUT_DIR / "label_mapping.json", "w") as f:
            json.dump(
                {"label2id": label2id, "id2label": id2label},
                f, indent=2
            )
        print("Saved best validation checkpoint.")

# ---------------- TEST BEST CHECKPOINT ----------------
del model
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True)

test_metrics = evaluate(test_loader)

test_predictions = test_metrics.pop("predictions")
test_targets = test_metrics.pop("targets")

print("\nFINAL E4 TEST METRICS")
print(json.dumps(test_metrics, indent=2))

report = classification_report(
    test_targets, test_predictions,
    labels=list(range(len(labels))),
    target_names=[id2label[i] for i in range(len(labels))],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    OUTPUT_DIR / "test_classification_report.csv"
)
pd.DataFrame(
    confusion_matrix(test_targets, test_predictions),
    index=[id2label[i] for i in range(len(labels))],
    columns=[id2label[i] for i in range(len(labels))]
).to_csv(OUTPUT_DIR / "test_confusion_matrix.csv")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

with open(OUTPUT_DIR / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

# ---------------- EXPORT ZIP ----------------
zip_path = shutil.make_archive(
    "/content/bert_tiny_e4_multilingual", "zip", OUTPUT_DIR
)
print("\nModel ZIP:", zip_path)
print("Model directory size (MB):",
      round(sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*") if p.is_file()) / 1e6, 2))

from google.colab import files
files.download(zip_path)

GPU: Tesla T4
Train: 370170
Validation: 42710
Test: 47120
Labels: {'as-IN': 0, 'bn-IN': 1, 'en-IN': 2, 'gu-IN': 3, 'hi-IN': 4, 'kn-IN': 5, 'ml-IN': 6, 'mr-IN': 7, 'none': 8, 'pa-IN': 9, 'ta-IN': 10, 'te-IN': 11}
Tokenizer vocabulary: 30522


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 19.3MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

...

Saved best validation checkpoint.


Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]


FINAL E4 TEST METRICS
{
  "loss": 0.0025145414953841607,
  "accuracy": 0.9996816638370118,
  "macro_f1": 0.9996887074004892
}

Model ZIP: /content/bert_tiny_e4_multilingual.zip
Model directory size (MB): 20.02


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import sys
import shutil
import subprocess
import getpass
from pathlib import Path
from IPython.display import display, Markdown

REPO_URL = (
    "https://github.com/"
    "abhinavnautiyalDS/multilingual-voice-language-classifier.git"
)
BRANCH = "finetune/e4-native-script-confusion"
REPO_DIR = Path("/content/multilingual-voice-language-classifier")

# 1. Verify the three source artifacts
dataset_candidates = [
    Path("/content/dataset_splits_audit1.zip"),
    Path("/content/dataset_splits_audit1 (1).zip"),
]
dataset_zip = next((p for p in dataset_candidates if p.exists()), None)

e2_zip = Path("/content/bert_tiny_e2_multilingual.zip")
e4_zip = Path("/content/bert_tiny_e4_multilingual.zip")

for p in [dataset_zip, e2_zip, e4_zip]:
    if p is None or not p.exists():
        raise FileNotFoundError(f"Required ZIP not found: {p}")

print("Artifacts ready:")
for p in [dataset_zip, e2_zip, e4_zip]:
    print(f"  {p.name}: {p.stat().st_size / 1e6:.2f} MB")

# 2. Install Git LFS
subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq git-lfs"],
    check=True
)
subprocess.run(["git", "lfs", "install"], check=True)

# 3. Clone the existing feature branch (or reuse its clone)
if not (REPO_DIR / ".git").exists():
    subprocess.run([
        "git", "clone",
        "--branch", BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(
        ["git", "fetch", "origin", BRANCH],
        cwd=REPO_DIR, check=True
    )
    subprocess.run(
        ["git", "checkout", BRANCH],
        cwd=REPO_DIR, check=True
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=REPO_DIR, check=True
    )

# 4. Copy artifacts into the documented repository structure
artifact_dir = REPO_DIR / "artifacts"
dataset_dir = artifact_dir / "datasets"
model_dir = artifact_dir / "models"
dataset_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(dataset_zip, dataset_dir / "dataset_splits_audit1.zip")
shutil.copy

Artifacts ready:
  dataset_splits_audit1.zip: 73.46 MB
  bert_tiny_e2_multilingual.zip: 16.57 MB
  bert_tiny_e4_multilingual.zip: 18.04 MB


<function shutil.copy(src, dst, *, follow_symlinks=True)>

In [ ]:
from pathlib import Path
import shutil

files = [
    Path("/content/dataset_splits_audit1.zip"),
    Path("/content/dataset_splits_audit1 (1).zip"),
    Path("/content/bert_tiny_e2_multilingual.zip"),
    Path("/content/bert_tiny_e4_multilingual.zip"),
]

for p in files:
    if p.exists():
        print(f"FOUND  {p} | {p.stat().st_size / 1e6:.2f} MB")

print("\nCopy function:", shutil.copy)
print("Repository exists:",
      Path("/content/multilingual-voice-language-classifier/.git").exists())

FOUND  /content/dataset_splits_audit1.zip | 73.46 MB
FOUND  /content/dataset_splits_audit1 (1).zip | 73.46 MB
FOUND  /content/bert_tiny_e2_multilingual.zip | 16.57 MB
FOUND  /content/bert_tiny_e4_multilingual.zip | 18.04 MB

Copy function: <function copy at 0x782737c2b380>
Repository exists: True


In [ ]:
import os
import shutil
import subprocess
import getpass
from pathlib import Path

REPO = Path("/content/multilingual-voice-language-classifier")
BRANCH = "finetune/e4-native-script-confusion"

DATASET = Path("/content/dataset_splits_audit1.zip")
E2 = Path("/content/bert_tiny_e2_multilingual.zip")
E4 = Path("/content/bert_tiny_e4_multilingual.zip")

def run(cmd, *, cwd=None, env=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

# 1. Validate sources and existing repository
assert (REPO / ".git").exists(), "Existing Git repository not found."
for p in [DATASET, E2, E4]:
    assert p.is_file(), f"Missing source file: {p}"

# 2. Install Git LFS if needed
if shutil.which("git-lfs") is None:
    run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq git-lfs"])

run(["git", "lfs", "install"])

# 3. Ensure we're on the intended branch and up to date
run(["git", "fetch", "origin", BRANCH], cwd=REPO)
run(["git", "checkout", BRANCH], cwd=REPO)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)

# 4. Copy files safely (avoid SameFileError)
destinations = [
    (DATASET, REPO / "artifacts/datasets/dataset_splits_audit1.zip"),
    (E2, REPO / "artifacts/models/bert_tiny_e2_multilingual.zip"),
    (E4, REPO / "artifacts/models/bert_tiny_e4_multilingual.zip"),
]

for src, dst in destinations:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() == dst.resolve():
        print("Already at destination:", dst)
    else:
        shutil.copy2(src, dst)
        print("Copied:", src.name, "->", dst.relative_to(REPO))

# 5. Confirm ZIPs are tracked by Git LFS
run(["git", "lfs", "track", "*.zip"], cwd=REPO)
run(["git", "add", ".gitattributes"], cwd=REPO)
run([
    "git", "add",
    "artifacts/datasets/dataset_splits_audit1.zip",
    "artifacts/models/bert_tiny_e2_multilingual.zip",
    "artifacts/models/bert_tiny_e4_multilingual.zip",
], cwd=REPO)

print("\nGit LFS files:")
run(["git", "lfs", "ls-files"], cwd=REPO)

print("\nGit status:")
run(["git", "status", "--short"], cwd=REPO)

# 6. Commit only if there are staged changes
changed = subprocess.run(
    ["git", "diff", "--cached", "--quiet"], cwd=REPO
).returncode != 0

if changed:
    run([
        "git", "config", "user.name", "Abhinav Nautiyal"
    ], cwd=REPO)
    run([
        "git", "config", "user.email",
        "164336356+abhinavnautiyalDS@users.noreply.github.com"
    ], cwd=REPO)
    run([
        "git", "commit", "-m",
        "Add dataset splits and E2 E4 model artifacts"
    ], cwd=REPO)
else:
    print("No staged changes; artifacts may already be committed.")

# 7. Push securely; token input stays hidden
token = getpass.getpass("GitHub PAT with repository Contents write access: ").strip()
if not token:
    raise ValueError("Token was empty.")

askpass = Path("/content/git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) printf "%s" "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    run(["git", "push", "origin", BRANCH], cwd=REPO, env=env)
    print("\nSUCCESS: push completed.")
finally:
    env.pop("GITHUB_TOKEN", None)
    token = None
    askpass.unlink(missing_ok=True)

print(
    "Verify files here:\n"
    "https://github.com/abhinavnautiyalDS/"
    "multilingual-voice-language-classifier/tree/"
    "finetune/e4-native-script-confusion/artifacts"
)

+ git lfs install
+ git fetch origin finetune/e4-native-script-confusion
+ git checkout finetune/e4-native-script-confusion
+ git pull --ff-only origin finetune/e4-native-script-confusion
Copied: dataset_splits_audit1.zip -> artifacts/datasets/dataset_splits_audit1.zip
Copied: bert_tiny_e2_multilingual.zip -> artifacts/models/bert_tiny_e2_multilingual.zip
Copied: bert_tiny_e4_multilingual.zip -> artifacts/models/bert_tiny_e4_multilingual.zip
+ git lfs track *.zip
+ git add .gitattributes
+ git add artifacts/datasets/dataset_splits_audit1.zip artifacts/models/bert_tiny_e2_multilingual.zip artifacts/models/bert_tiny_e4_multilingual.zip

Git LFS files:
+ git lfs ls-files

Git status:
+ git status --short
+ git config user.name Abhinav Nautiyal
+ git config user.email 164336356+abhinavnautiyalDS@users.noreply.github.com
+ git commit -m Add dataset splits and E2 E4 model artifacts
GitHub PAT with repository Contents write access: ··········
+ git push origin finetune/e4-native-script-confus

In [ ]:
import pandas as pd

CSV_PATH = (
    "/content/data_plus_synthetic_updated.csv")

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (460000, 12)


,text,label,category,source,source_file,source_group_id,text_sha256,review_status,is_mixed_script_candidate,is_synthetic,generator,template_id
0,Climate-conscious consumers ആയതിനാൽ ഇന്ന് sust...,ml-IN,malayalam_english_mixed_script,ai4bharat/IndicCMix,ml.parquet,indiccmix:ml:ml_551054,f62ab8dc6f4eb33c04b865c0deb3af42e4cd14359d1c9b...,candidate_unreviewed,True,False,NaN,NaN
1,"ফেস্টিভ্যালের সময়, পঞ্চমুখী ভাদ্যম পারফর্ম্যা...",bn-IN,bengali_only,ai4bharat/IndicCMix,bn.parquet,indiccmix:bn:bn_117667,235d843dd4649e88a55f2545743caec4d4747a6fed42cb...,candidate_unreviewed,False,False,NaN,NaN
2,ভাৰতৰ handloom weavers সকলে Facebook marketpla...,as-IN,assamese_english_mixed_script,ai4bharat/IndicCMix,as.parquet,indiccmix:as:as_023627,4443942959918fd02daabb7be2198e63c5c6916eff3f4e...,candidate_unreviewed,True,False,NaN,NaN
3,இயக்குனரகத்தின் ஆபீசர்ஸ் களின் கடுமையான இன்ஸ்ப...,ta-IN,tamil_only,ai4bharat/IndicCMix,ta.parquet,indiccmix:ta:ta_1037914,75e4a67967862953a8b1338d4e43a815ee5aead6f644c3...,candidate_unreviewed,False,False,NaN,NaN
4,ভাৰতৰ ৰাশিয়া সৈতে হাইডোৰপাওয়াৰ ইনোভেশন্স ৰ ক্ষ...,as-IN,assamese_only,ai4bharat/IndicCMix,as.parquet,indiccmix:as:as_050621,fd422e1711d39604575ad03a2b01fc708ed72d75926fbf...,candidate_unreviewed,False,False,NaN,NaN


In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/content/multilingual-voice-language-classifier")
BRANCH = "finetune/e4-native-script-confusion"

csv_path = REPO / "data/raw/data_plus_synthetic_updated.csv"

assert csv_path.exists(), f"CSV not found: {csv_path}"

print("CSV size (MB):", round(csv_path.stat().st_size / (1024 * 1024), 2))

# Check Git LFS for large CSV files
subprocess.run(
    ["git", "lfs", "track", "*.csv"],
    cwd=REPO,
    check=True
)

subprocess.run(
    ["git", "add", ".gitattributes", "data/raw/data_plus_synthetic_updated.csv"],
    cwd=REPO,
    check=True
)

subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO,
    check=True
)

print("\nAb commit aur push ke liye ready hai.")

AssertionError: CSV not found: /content/multilingual-voice-language-classifier/data/raw/data_plus_synthetic_updated.csv